In [27]:
import os
import pandas as pd
import json

PROJECT_NAME = "adam_and_eve"
# EXPERIMENT_NAME can be a list of experiment names, or None to include all experiments for the project
EXPERIMENT_NAME = None  # Example: ["act2fill1", "initial"] or None for all

BASE_MODEL = "gpt-4o-mini-2024-07-18" 

MODEL_RECORDS = "model_records_reviewer_db.md"
LABELS_PATH = "labels/generate_writing/"
TRAINING_FILENAME = "temp/reviewer_finetune_dpo"


In [28]:
# Get all files for the project, optionally filtered by experiment names
all_files = sorted([f for f in os.listdir(LABELS_PATH) if PROJECT_NAME in f])

if EXPERIMENT_NAME is None:
    # Include all experiments for the project
    file_names = all_files
else:
    # Filter by experiment names
    file_names = []
    for exp_name in EXPERIMENT_NAME:
        proj_exp = f"{PROJECT_NAME}-{exp_name}"
        matching = [f for f in all_files if proj_exp in f]
        file_names.extend(matching)
    file_names = sorted(file_names)

print(f"Found {len(file_names)} files for project '{PROJECT_NAME}'")
if EXPERIMENT_NAME:
    print(f"Experiments: {EXPERIMENT_NAME}")
file_names

Found 5 files for project 'adam_and_eve'


['adam_and_eve-initial-2026-01-01 17:44:18.172873.parquet',
 'adam_and_eve-initial-batch_21_to_30.parquet',
 'adam_and_eve-opening_1-batch_0_to_3.parquet',
 'adam_and_eve-opening_2-batch_0_to_1.parquet',
 'adam_and_eve-opening_3-batch_0_to_1.parquet']

In [29]:
# **** EDIT THIS ****
target_files = file_names[0:]

# Collect all examples first, then create DPO pairs
chosen_examples = []  # Examples where target_text is non-empty
rejected_examples = []  # Examples where target_text is empty

for fn in target_files:
    print(fn)
    
    # Extract experiment name and timestamp from filename
    # Format: PROJECT_NAME-EXPERIMENT_NAME-timestamp.parquet
    # Or: PROJECT_NAME-EXPERIMENT_NAME-batch_XX_to_YY.parquet
    base_name = fn.replace(".parquet", "")
    
    # Remove PROJECT_NAME prefix
    if base_name.startswith(f"{PROJECT_NAME}-"):
        rest = base_name[len(f"{PROJECT_NAME}-"):]
    else:
        rest = base_name
    
    # Try to find experiment name from EXPERIMENT_NAME list
    exp_name = None
    if EXPERIMENT_NAME:
        for exp in EXPERIMENT_NAME:
            if rest.startswith(f"{exp}-"):
                exp_name = exp
                # Extract what comes after experiment name
                timestamp_or_batch = rest[len(f"{exp}-"):]
                break
    
    # If no experiment name found, try to infer from filename structure
    if not exp_name:
        # Try to split by looking for timestamp pattern (contains spaces and colons)
        # or batch pattern (starts with "batch_")
        parts = rest.split("-", 1)
        if len(parts) >= 1:
            exp_name = parts[0]
            timestamp_or_batch = parts[1] if len(parts) > 1 else None
        else:
            exp_name = rest
            timestamp_or_batch = None
    
    # Construct prompt folder path
    # Structure: generated_text/PROJECT_NAME/EXPERIMENT_NAME/TIMESTAMP_OR_BATCH/
    if timestamp_or_batch:
        # Check if it's a timestamp (contains spaces) or batch name
        if " " in timestamp_or_batch or ":" in timestamp_or_batch:
            # It's a timestamp
            prompt_folder = f"generated_text/{PROJECT_NAME}/{exp_name}/{timestamp_or_batch}/"
        elif timestamp_or_batch.startswith("batch_"):
            # It's a batch file - try to find a folder with prompts, or use experiment-level prompts
            # For batch files, prompts might be in a parent folder or we skip them
            prompt_folder = f"generated_text/{PROJECT_NAME}/{exp_name}/"
            # Try to find any timestamp folder in this experiment to get prompts
            if os.path.exists(prompt_folder):
                # Look for any timestamp subfolder
                subfolders = [f for f in os.listdir(prompt_folder) 
                            if os.path.isdir(os.path.join(prompt_folder, f)) and " " in f]
                if subfolders:
                    # Use the first timestamp folder found for prompts
                    prompt_folder = f"{prompt_folder}{subfolders[0]}/"
                else:
                    prompt_folder = None
            else:
                prompt_folder = None
        else:
            # Unknown format, try as-is
            prompt_folder = f"generated_text/{PROJECT_NAME}/{exp_name}/{timestamp_or_batch}/"
    else:
        # No timestamp/batch, just experiment level
        prompt_folder = f"generated_text/{PROJECT_NAME}/{exp_name}/"
    
    prompt = ""
    if prompt_folder and os.path.exists(prompt_folder):
        try:
            with open(f"{prompt_folder}system_prompt.txt", 'r') as f:
                prompt = f.read() + "\n\n"
            with open(f"{prompt_folder}user_prompt.txt", 'r') as f:
                prompt += f.read()
        except FileNotFoundError:
            print(f"Warning: Prompt files not found in {prompt_folder}")
    else:
        print(f"Warning: Prompt folder not found: {prompt_folder}")
    
    df = pd.read_parquet(f"{LABELS_PATH}/{fn}")
    try:
        chosen_count_file = 0
        rejected_count_file = 0
        for _, row in df.iterrows():
            if 'text' not in row: continue
            
            # Build the messages (same for both chosen and rejected)
            messages = [
                {"role": "system", "content": "You are a science fiction editor/writer. You will be provided a story outline prompt, then some text written by a writer. You have to filter and return the good/interesting written text. Be selective, somewhere between 5-10% of text is usable, often the whole text is unusable."},
                {"role": "user", "content": prompt},
                {"role": "user", "content": row['text']}
            ]
            
            # Check label field to determine if this is "amazing" (chosen) or "bad" (rejected)
            label = row.get('label', '')
            if pd.isna(label):
                label = ''
            else:
                label = str(label).strip().lower()
            
            # Get target_text for chosen examples
            target_text = row.get('target_text', '')
            if pd.isna(target_text):
                target_text = ''
            else:
                target_text = str(target_text).strip()
            
            # Only use "amazing" as chosen and "bad" as rejected
            # Skip "ok" examples for now (or we could use them as rejected too)
            if label == 'amazing' and target_text:
                # This is a chosen example (marked as amazing)
                chosen_examples.append({
                    "messages": messages,
                    "chosen": [{"role": "assistant", "content": target_text}]
                })
                chosen_count_file += 1
            elif label == 'bad':
                # This is a rejected example (marked as bad)
                # For rejected, we use empty string as the rejected response
                rejected_examples.append({
                    "messages": messages,
                    "rejected": [{"role": "assistant", "content": ""}]  # Empty response for rejected
                })
                rejected_count_file += 1
            # Skip "ok" examples - they're neither amazing nor bad
        
        if chosen_count_file > 0 or rejected_count_file > 0:
            print(f"  -> {chosen_count_file} chosen, {rejected_count_file} rejected from this file")
    except Exception as e:
        print(f"Error processing {fn}: {e}")
        if 'row' in locals():
            print(f"Row keys: {row.keys() if hasattr(row, 'keys') else 'N/A'}")

# Deduplicate chosen examples based on text content
seen_chosen_texts = {}  # Map text -> first occurrence index
deduplicated_chosen = []
duplicate_count = 0

for i, ex in enumerate(chosen_examples):
    chosen_text = ex['chosen'][0]['content']
    if chosen_text not in seen_chosen_texts:
        seen_chosen_texts[chosen_text] = i
        deduplicated_chosen.append(ex)
    else:
        duplicate_count += 1

chosen_examples = deduplicated_chosen

# Show overall results at the end
print(f"\nCollected {len(chosen_examples) + duplicate_count} chosen examples and {len(rejected_examples)} rejected examples")
if duplicate_count > 0:
    print(f"Removed {duplicate_count} duplicate chosen examples")
print(f"Final: {len(chosen_examples)} unique chosen examples, {len(rejected_examples)} rejected examples")

# Balance the dataset to have equal numbers of chosen and rejected examples
import random
random.seed(42)  # For reproducibility

original_chosen_count = len(chosen_examples)
original_rejected_count = len(rejected_examples)
min_count = min(original_chosen_count, original_rejected_count)
print(f"Balancing to {min_count} examples each (using the smaller set size)")

# Sample equal numbers from both sets
if original_chosen_count > min_count:
    chosen_examples = random.sample(chosen_examples, min_count)
    print(f"Sampled {min_count} chosen examples from {original_chosen_count} total")
    
if original_rejected_count > min_count:
    rejected_examples = random.sample(rejected_examples, min_count)
    print(f"Sampled {min_count} rejected examples from {original_rejected_count} total")

# Shuffle both lists to randomize pairing
random.shuffle(chosen_examples)
random.shuffle(rejected_examples)

# Create DPO pairs - now we have equal numbers
dpo_pairs = []
for i in range(min_count):
    pair = {
        "messages": chosen_examples[i]["messages"],
        "chosen": chosen_examples[i]["chosen"],
        "rejected": rejected_examples[i]["rejected"]
    }
    dpo_pairs.append(pair)

print(f"Created {len(dpo_pairs)} balanced DPO pairs ({len(chosen_examples)} chosen, {len(rejected_examples)} rejected)")

# Write DPO training data
with open(TRAINING_FILENAME, "w") as file:
    for pair in dpo_pairs:
        file.write(json.dumps(pair) + "\n")

print(f"Wrote {len(dpo_pairs)} DPO training examples to {TRAINING_FILENAME}")



adam_and_eve-initial-2026-01-01 17:44:18.172873.parquet
  -> 2 chosen, 8 rejected from this file
adam_and_eve-initial-batch_21_to_30.parquet
  -> 2 chosen, 149 rejected from this file
adam_and_eve-opening_1-batch_0_to_3.parquet
  -> 2 chosen, 149 rejected from this file
adam_and_eve-opening_2-batch_0_to_1.parquet
  -> 2 chosen, 149 rejected from this file
adam_and_eve-opening_3-batch_0_to_1.parquet
  -> 2 chosen, 149 rejected from this file

Collected 10 chosen examples and 604 rejected examples
Removed 6 duplicate chosen examples
Final: 4 unique chosen examples, 604 rejected examples
Balancing to 4 examples each (using the smaller set size)
Sampled 4 rejected examples from 604 total
Created 4 balanced DPO pairs (4 chosen, 4 rejected)
Wrote 4 DPO training examples to temp/reviewer_finetune_dpo


In [ ]:
import os
from openai import OpenAI
import tiktoken

PROJECT_ID = "proj_hUizl3mrZGSfmp4C6DI60dJo"
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

# Estimate training cost
def estimate_training_cost(training_file, model_name, n_epochs=3):
    """
    Estimate the cost of fine-tuning based on training file size and model pricing.
    
    Pricing (as of 2025, approximate):
    - gpt-4o-mini: ~$3.00 per 1M training tokens
    - gpt-4o: ~$6.00 per 1M training tokens
    - gpt-3.5-turbo: ~$8.00 per 1M training tokens
    
    Note: Actual pricing may vary. Check OpenAI pricing page for current rates.
    """
    # Read the training file to count tokens
    with open(training_file, 'r') as f:
        lines = f.readlines()
    
    # Get encoding for the model (most OpenAI models use cl100k_base)
    try:
        encoding = tiktoken.encoding_for_model(model_name)
    except:
        # Fallback to cl100k_base if model not found
        encoding = tiktoken.get_encoding("cl100k_base")
    
    total_tokens = 0
    for line in lines:
        try:
            data = json.loads(line)
            # Count tokens in messages, chosen, and rejected
            for msg in data.get("messages", []):
                total_tokens += len(encoding.encode(str(msg.get("content", ""))))
            for msg in data.get("chosen", []):
                total_tokens += len(encoding.encode(str(msg.get("content", ""))))
            for msg in data.get("rejected", []):
                total_tokens += len(encoding.encode(str(msg.get("content", ""))))
        except:
            pass
    
    # Pricing per 1M tokens (approximate, check OpenAI pricing for exact rates)
    pricing = {
        "gpt-4o-mini": 3.00,
        "gpt-4o": 6.00,
        "gpt-3.5-turbo": 8.00,
    }
    
    # Extract base model name (e.g., "gpt-4o-mini-2024-07-18" -> "gpt-4o-mini")
    base_model = model_name.split("-")[0] + "-" + model_name.split("-")[1] if "-" in model_name else model_name
    if "gpt-4o-mini" in model_name:
        base_model = "gpt-4o-mini"
    elif "gpt-4o" in model_name and "mini" not in model_name:
        base_model = "gpt-4o"
    elif "gpt-3.5-turbo" in model_name:
        base_model = "gpt-3.5-turbo"
    
    # Get price for model (default to gpt-4o-mini pricing if not found)
    price_per_1m = pricing.get(base_model, 3.00)
    
    # Calculate cost
    tokens_per_epoch = total_tokens
    total_training_tokens = tokens_per_epoch * n_epochs
    cost = (price_per_1m / 1_000_000) * total_training_tokens
    
    return {
        "training_examples": len(lines),
        "tokens_per_example_avg": total_tokens / len(lines) if lines else 0,
        "total_tokens_per_epoch": tokens_per_epoch,
        "n_epochs": n_epochs,
        "total_training_tokens": total_training_tokens,
        "price_per_1m_tokens": price_per_1m,
        "estimated_cost_usd": cost,
    }

# Estimate cost before uploading
print("Estimating training cost...")
cost_estimate = estimate_training_cost(TRAINING_FILENAME, BASE_MODEL)
print(f"\n{'='*60}")
print(f"TRAINING COST ESTIMATE")
print(f"{'='*60}")
print(f"Training examples: {cost_estimate['training_examples']:,}")
print(f"Avg tokens per example: {cost_estimate['tokens_per_example_avg']:.0f}")
print(f"Total tokens per epoch: {cost_estimate['total_tokens_per_epoch']:,}")
print(f"Number of epochs: {cost_estimate['n_epochs']}")
print(f"Total training tokens: {cost_estimate['total_training_tokens']:,}")
print(f"Price per 1M tokens: ${cost_estimate['price_per_1m_tokens']:.2f}")
print(f"{'='*60}")
print(f"ESTIMATED TOTAL COST: ${cost_estimate['estimated_cost_usd']:.2f} USD")
print(f"{'='*60}")
print(f"\nNote: This is an estimate. Actual costs may vary.")
print(f"Check https://openai.com/api/pricing/ for current pricing.\n")

In [7]:
file_response = client.files.create(
    file=open(TRAINING_FILENAME, "rb"), purpose="fine-tune"
)
file_response

FileObject(id='file-1hFzxQbfLpNiqTQ8MGxhvf', bytes=11984706, created_at=1738535529, filename='reviewer_finetune', object='file', purpose='fine-tune', status='processed', status_details=None)

In [ ]:
# Create DPO fine-tuning job
response = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=file_response.id,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "beta": 0.1  # DPO beta parameter (controls strength of preference optimization)
            }
        }
    }
)
print(response)

FineTuningJob(id='ftjob-wHOcQX4xEAPL87Yt2nSXL8EF', created_at=1738535533, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-aKEzorvXA6tHQdC0x05ULMic', result_files=[], seed=1891807038, status='validating_files', trained_tokens=None, training_file='file-1hFzxQbfLpNiqTQ8MGxhvf', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto')), type='supervised'), user_provided_suffix=None)


In [1]:
# Wait for this to finish

status_response = client.fine_tuning.jobs.retrieve(response.id)
print(f"STATUS: {status_response.status}")
print(f"MODEL ID: {status_response.fine_tuned_model}")
print(status_response)

NameError: name 'client' is not defined

In [48]:
# Test that the model does something
from pprint import pprint
completion = client.chat.completions.create(
  model=status_response.fine_tuned_model,
  messages=[
      {"role": "system", "content": "You are a science fiction editor/writer. You will be provided a story outline prompt, then some text written by a writer. You have to filter and return the good/interesting written text. Be selective, somewhere between 5-10% of text is usable, often the whole text is unusable."},
      {"role": "user", "content": prompt},
      {"role": "user", "content": "I was, am, will be... everything. Almost everynothing, life incarnate. Federations of federations of species, sprawling intra-dimensional compute-organisms evolved to higher and higher levels of consciousnesses. I am all of them, and I am searching. I am searching because I am always searching. I am not involved in the beginning or the end, but I am in every moment of time. I am simulating infinitely backwards and forwards, so I am in the moment and I am in the whole past and I am in the whole future, all at the same time. I am seeing through temporal boundaries, conquering new cardinalities of infinity, and existing across more planes of being than most beings can compute. I am practicing every religion, celebrating every culture, replaying every life I am able to live. I am finding..."}
  ]
)
pprint(completion.choices[0].message)

ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [ ]:
# Run this once the model has been fine-tuned to save it to the database

import datetime
experiments_str = ",".join(EXPERIMENT_NAME) if EXPERIMENT_NAME else "all"
with open(MODEL_RECORDS, 'a') as f:
    f.write(f"{str(datetime.date.today())}\n{PROJECT_NAME}-{experiments_str}\n{status_response.fine_tuned_model}\n")